# Build an Adaptive Multi-Agent AI System with ReAct Pattern

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/multi-agent-workflows/tutorial_react_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Learn to build a ReAct (Reasoning + Acting) agent system that adaptively plans and executes tasks through iterative reasoning and reflection.

**Pattern:** Loop of Thought → Action → Observation → Reflection until goal achieved

```
User: "Find France's GDP and calculate 5% of it"
  ↓
Step 1: Think "Need to search for France's GDP"
        Act: web_search → Observe: "€2.6 trillion"
        Reflect: "Got the GDP, now need to calculate percentage"
  ↓
Step 2: Think "Calculate 5% of 2.6 trillion"
        Act: math → Observe: "130 billion"
        Reflect: "Goal achieved with final answer"
```

**Key concepts:** Adaptive planning, iterative reasoning, self-reflection, dynamic task decomposition.

**vs Planner Pattern:** ReAct decides next step based on observations, Planner creates full plan upfront.

---

## Setup

Project structure: `agents/`, `tools/`, `workflows/`, `utils/`

**Configuration:** Shared Flyte environment with Docker image + secrets. Agents inherit this config but can override for custom resources.

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/multi-agent-workflows/
    !uv pip install -r requirements.txt
    
# this is just for viewing the code files within the notebook
from utils.file_viewer import view_file

In [3]:
view_file("requirements.txt")

In [4]:
view_file("config.py")

## Connect to Flyte Cluster
You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.

Flyte gives you...

- If you don't have a Flyte cluster you can request demo access by filling out the form [here](https://flyte.org/).
- If you already have a Flyte cluster, you can connect to it by setting your endpoint in the Flyte configuration.


In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --auth-type headless\
    --builder remote \
    --domain development \
    --project flytesnacks

You can now adjust the configuration by modifying the `.flyte/config.yaml` file.

In [4]:
view_file(".flyte/config.yaml")

## Set your API Key(s)

The project is setup to read in secrets from a `.env` file.

You can create this file in the root of this tutorial `tutorials/multi-agent-workflows` and add your API keys there.

But if you prefer to just enter a key once in this notebook you can run the cell below:

In [ ]:
# Skip if API key is already set in .env or environment
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on the remote Flyte cluster, add the API keys as secrets. 

You can skip this step if you're not using a Flyte cluster and only want to run the examples locally.


In [ ]:
# run this and enter your API key as the input
!flyte create secret OPENAI_API_KEY

## Run the Agent

At this point you should be setup to run the ReAct agent. 

I suggest giving it a try before we walk through the code in the next section.

**Run locally:**

In [5]:
!python -m workflows.react --request "Find France's GDP and calculate 5% of it" --local

Running workflow LOCALLY with flyte.init()

=== ReAct Multi-Agent Workflow ===
Goal: Find France's GDP and calculate 5% of it
Max steps: 10

[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] ================================================================================
[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] ReAct WORKFLOW - Goal: Find France's GDP and calculate 5% of it
[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] ================================================================================
[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] 
[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] STEP 1
[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] ================================================================================
[68708bbf-797b-4690-8877-79e326fb9c0a][d0ks8befx9xxjsud8ia28zel4] 
[ReAct] Reasoning about next action...
[68708bbf-797b-4690-8877-79e326fb9c0a][d0k

**Run on the remote Flyte cluster:**

The first time running the agent a container image will be built and pushed to the Flyte cluster.

This may take some time depending on the size of your dependencies.

In [6]:
!python -m workflows.react --request "Find France's GDP and calculate 5% of it" 

Running workflow REMOTELY with flyte.init_from_config()

=== ReAct Multi-Agent Workflow ===
Goal: Find France's GDP and calculate 5% of it
Max steps: 10

23:12:52.713983 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
23:12:52.716034 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: rxkxkrd2wspbczk6642v
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/rxkxkrd2wspbczk6642v



# Code Walkthrough

Let's walk through the code to understand how the planner agent is structured and how it works.

We'll cover all the key file types, but you can explore all the agents and tools in their respective folders.

## Infrastructure

**Decorators** - Registration system for agents and tools. 

This allows for easy addition and management of new agents and tools within the workflow.

In [ ]:
view_file("agents/planner_agent.py")

---

## Orchestrator - The Agentic Workflow

Executes plans with dependency-aware parallelism.

**Flow:**
1. Get plan from planner
2. Loop: Find steps with satisfied dependencies → Execute in parallel → Mark complete
3. For dependent steps: Inject previous results via `build_task_with_context()`

**Context passing:**
```python
# Step 2 depends on steps 0 and 1
task = """
RESULTS FROM PREVIOUS STEPS:
  - Step 0 (math): 5
  - Step 1 (math): 11

YOUR TASK:
Add the results
"""
```

**Key features:** Automatic parallelization (`asyncio.gather`), result propagation, circular dependency detection.

In [ ]:

view_file("workflows/planner.py")

## ReAct Orchestrator - Adaptive Execution Loop

The ReAct pattern uses an iterative loop instead of upfront planning.

**Flow:**
1. **Thought:** LLM reasons about what to do next based on goal + context
2. **Action:** Execute ONE agent with specific task
3. **Observation:** Capture result from agent
4. **Reflection:** LLM analyzes if result was helpful and if closer to goal
5. Repeat until goal achieved or max steps

**Key differences from Planner:**
- **Adaptive:** Decides next step based on observations, not fixed plan
- **Sequential:** One agent at a time (vs parallel waves)
- **Reflective:** Evaluates progress after each step
- **Flexible:** Can change strategy mid-execution

**Context management:**
```python
# Last 3 steps included in reasoning prompt
history_text = "\n\n".join([
    f"Step {s['step']}: {s['thought']}\n"
    f"Action: {s['action_agent']} - {s['action_task']}\n"
    f"Result: {s['observation']}\n"
    f"Reflection: {s['reflection']}"
    for s in context_history[-3:]
])
```

**Agent routing:** Uses `agent_registry` for dynamic dispatch (same as planner).

In [13]:
view_file("workflows/react.py")

---

## Running the Workflow

**Local (development):**
```bash
python -m workflows.react --local --request "your goal" --max-steps 10
```
In-process execution, fast iteration.

**Remote (production):**
```bash
python -m workflows.react --request "your goal" --max-steps 10
```
Distributed Flyte cluster, scalable and observable.

**Try these:**
- Simple: `"Calculate 5 factorial"`
- Multi-step: `"Find France's GDP and calculate 5% of it"`
- Complex: `"Search for Seattle's population, calculate the square root, then convert to words"`

In [ ]:
!python -m workflows.react --request "Find France's GDP in 2023, then calculate 5% of that value" --local --max-steps 5

In [16]:
!python -m workflows.react --request "Get weather for 3 cities, compare temperatures, and make an ASCII chart" --local --max-steps 10

Running workflow LOCALLY with flyte.init()

=== ReAct Multi-Agent Workflow ===
Goal: Get weather for 3 cities, compare temperatures, and make an ASCII chart
Max steps: 10

[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] ================================================================================
[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] ReAct WORKFLOW - Goal: Get weather for 3 cities, compare temperatures, and make an ASCII chart
[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] ================================================================================
[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] 
[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] STEP 1
[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] ================================================================================
[f84bc0e1-99da-40c3-a02d-c85b3ea42444][1x8x82ru18q1th4gs5qaav3y2] 
[ReAct] Reasoning a

In [17]:
!python -m workflows.react --request "Get weather for 3 cities, compare temperatures, and make an ASCII chart" --max-steps 10

Running workflow REMOTELY with flyte.init_from_config()

=== ReAct Multi-Agent Workflow ===
Goal: Get weather for 3 cities, compare temperatures, and make an ASCII chart
Max steps: 10

17:28:44.365498 WARNING  remote_builder.py:95 -  Image                          
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8 found. Skip    
                         building.                                              
17:28:44.369207 WARNING  _deploy.py:376 -  Built Image for environment base_env,
                         image:                                                 
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-a6e80122ad3ee2d24341681f57f3b7a8                

Execution: r4cvj8fqpkq82btjv6p9
URL: https://demo.hosted.unionai.cloud/v2/domain/development/project/flytesnacks/runs/r4cvj8fqpkq82btjv6p9



---

## Key Takeaways

**ReAct Pattern:**
- **Adaptive:** Decides next step based on observations
- **Reflective:** Self-evaluates progress after each action
- **Flexible:** Can change strategy based on results
- **Sequential:** One step at a time with full context

**When to use ReAct vs Planner:**
- **ReAct:** Unknown number of steps, need to adapt based on results, exploratory tasks
- **Planner:** Clear task decomposition, maximize parallelism, predictable workflows

**Architecture benefits:**
- Same agents/tools work with both patterns
- Type-safe, observable, scalable with Flyte
- Easy to add new orchestration patterns

**What makes this powerful:**
The LLM can adapt its strategy based on what it learns at each step, making it more flexible than static planning.

**Next steps:**
1. Try different prompts and observe reasoning
2. Adjust max_steps to see impact
3. Compare same task with planner vs ReAct
4. Implement your own orchestration pattern!

---

## Resources

- Full code: `tutorials/multi-agent-workflows/`
- Planner tutorial: [tutorial_planner_agent.ipynb](tutorial_planner_agent.ipynb)
- Flyte docs: https://docs.flyte.org
- ReAct paper: ["ReAct: Synergizing Reasoning and Acting in Language Models"](https://arxiv.org/abs/2210.03629)
- Questions? Join the Flyte community Slack!